# LEVIR-Ship — YOLOv8-P2 baseline vs 3 Adaptive-SRM variants
Notebook độc lập với `Levir_ship_training.ipynb` (phía mmdet): cùng dùng lại
`data/levir_ship_data.zip` đã có trên Hugging Face, nhưng code/kiến trúc là YOLO
(Ultralytics), không cần MMCV/install.sh. Chi tiết kiến trúc 3 biến thể ASRM: xem
`model_cfg/README.md` bên trong gói code.

Setup (pretrained `yolov8n.pt`, imgsz 512, batch 8, 3 seed 42/43/44) được chỉnh để khớp
baseline yolov8n LEVIR-Ship tham chiếu ở
[`reference implementation`](the reference implementation)
(`train_all_levir_yolov8n_p2_routing.py`), chỉ khác kiến trúc P2 (ASRM thay vì
DBSS/GCTS). Data dùng đúng fixed random crop split seed 42, train/val/test =
2320/788/788 như repo tham chiếu; cả ba training seed dùng chung split.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile

HF_REPO = 'HoangTrungNguyen/Levir_ship_training'
HF_READ_TOKEN = 'PASTE_READ_TOKEN_HERE'
HF_WRITE_TOKEN = 'PASTE_WRITE_TOKEN_HERE'  # chỉ dùng nếu UPLOAD_RESULTS=True

RUNTIME = Path('/marimo/Levir_ship_training_YOLO_runtime')
CODE_DIR = RUNTIME / 'code'
DATA_DIR = RUNTIME / 'data'          # tái dùng đúng data/levir_ship_data.zip của phía mmdet
YOLO_DATA_DIR = RUNTIME / 'yolo_data'  # layout images/labels/data.yaml build từ DATA_DIR
WORK_DIR = RUNTIME / 'runs_512px_pretrained_fixedsplit42_reference'
RESULT_CSV = RUNTIME / 'results' / 'comparison_512px_pretrained_fixedsplit42_reference.csv'
VENV_DIR = '/marimo/yolo-venv'
TRAIN_PYTHON = f'{VENV_DIR}/bin/python'

# Khớp baseline yolov8n LEVIR-Ship tham chiếu (reference implementation): pretrained
# yolov8n.pt, imgsz 512, batch 8, 3 seed 42/43/44 -- xem ghi chú ở cell trên.
PRETRAINED = 'yolov8n.pt'
EPOCHS, BATCH_SIZE, WORKERS, IMAGE_SIZE = 100, 8, 4, 512
SEEDS = (42, 43, 44)
SPLIT_SEED = 42  # cố định cho data; độc lập với training seed
PATIENCE = 20
# reference GAP+FTAL k15 foundation, with ASRM-guided P2' replacing plain P2.
CASES = ('baseline_gap_factorized_k15_nomosaic',)

RUN_SMOKE_TEST = True
RUN_TRAINING = True
FORCE_RERUN = False
UPLOAD_RESULTS = True
RUNTIME.mkdir(parents=True, exist_ok=True)


## 1. Tải code + tái dùng data đã có
`code/Levir_ship_training_YOLO_code.zip` chứa `ultralytics/` (bản local có 3 module
ASRM, không phải `pip install ultralytics`), `model_cfg/`, các script train/collect và
`requirements.txt`. Dữ liệu (`data/levir_ship_data.zip`, ~860MB) là **file đã có sẵn**
trên HF dùng chung với phía mmdet — không upload/tải lại bản sao thứ hai.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U',
                'huggingface_hub>=0.24'], check=True)
from huggingface_hub import hf_hub_download

token = None if HF_READ_TOKEN == 'PASTE_READ_TOKEN_HERE' else HF_READ_TOKEN
artifacts = {
    'code/Levir_ship_training_YOLO_code.zip': (RUNTIME / 'downloads/code.zip', CODE_DIR),
    'data/levir_ship_data.zip': (RUNTIME / 'downloads/data.zip', DATA_DIR),
}
for remote, (archive, destination) in artifacts.items():
    archive.parent.mkdir(parents=True, exist_ok=True)
    cached = hf_hub_download(repo_id=HF_REPO, repo_type='dataset',
                             filename=remote, token=token)
    shutil.copy2(cached, archive)
    if FORCE_RERUN and destination.exists():
        shutil.rmtree(destination)
    if not destination.exists():
        destination.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(destination)
print('CODE_DIR =', CODE_DIR)
print('DATA_DIR =', DATA_DIR)


## 2. Venv riêng cho training
Không cần MMCV. Cell dưới cài tường minh PyTorch 2.11 CUDA 13.0 vào venv trước, rồi
mới cài các dependency còn lại. Cách này tránh pip tự chọn CPU wheel khiến
`torch.cuda.is_available() == False`. Venv độc lập với kernel Marimo.

In [ ]:
if not Path(TRAIN_PYTHON).is_file():
    subprocess.run([sys.executable, '-m', 'venv', VENV_DIR], check=True)
subprocess.run([TRAIN_PYTHON, '-m', 'pip', 'install', '-U', 'pip'], check=True)
cuda_probe = subprocess.run(
    [TRAIN_PYTHON, '-c', 'import torch; raise SystemExit(0 if torch.cuda.is_available() else 1)'],
    check=False,
)
if cuda_probe.returncode != 0:
    subprocess.run(
        [TRAIN_PYTHON, '-m', 'pip', 'install', '--force-reinstall',
         'torch==2.11.0', 'torchvision==0.26.0', 'torchaudio==2.11.0',
         '--index-url', 'https://download.pytorch.org/whl/cu130'],
        check=True,
    )
subprocess.run([TRAIN_PYTHON, '-m', 'pip', 'install', '-r',
                str(CODE_DIR / 'requirements.txt')], check=True)
subprocess.run([TRAIN_PYTHON, '-c',
    'import torch; assert torch.cuda.is_available(); print("torch/cuda:", torch.__version__, torch.version.cuda); print("GPU:", torch.cuda.get_device_name(0))'],
    check=True)


## 3. Build layout Ultralytics (images/labels/data.yaml)
Tạo đúng fixed split của `reference implementation`: sort toàn bộ 3.896 crop stem,
shuffle bằng `random.Random(42)`, rồi cắt train/val/test = 2320/788/788. Ảnh và
label YOLO gốc (`LevirShipData/All Images` + `All Annotations`) được symlink nên
không copy lại ~860MB ảnh. `split_manifest.json` lưu stem để kiểm tra.

In [ ]:
subprocess.run([TRAIN_PYTHON, str(CODE_DIR / 'prepare_yolo_data.py'),
                '--data-root', str(DATA_DIR), '--out-root', str(YOLO_DATA_DIR),
                '--split-seed', str(SPLIT_SEED)],
               check=True)
DATA_YAML = YOLO_DATA_DIR / 'data.yaml'
print((DATA_YAML).read_text())


## 4. Smoke test kiến trúc
Build cả 4 case (baseline P2 + 3 biến thể ASRM), forward+backward trên tensor giả — bắt
lỗi cấu hình trước khi tốn thời gian train thật.

In [ ]:
if RUN_SMOKE_TEST:
    subprocess.run([TRAIN_PYTHON, str(CODE_DIR / 'smoke_matrix.py'),
                    '--device', 'cuda'], check=True)


## 5. Train + test tuần tự 5 case × 3 seed
Mỗi (case, seed): `run_experiment.py` transfer-learn từ `PRETRAINED` (remap layer index
cho các biến thể có `nn.Identity` raw-image tap, xem `model_cfg/README.md`),
train qua `ultralytics.YOLO(...).train()` rồi `.val(split='test')`, ghi
`test_metrics.json` vào `WORK_DIR/<case>/seed_<seed>/`.

In [ ]:
if RUN_TRAINING:
    for seed in SEEDS:
        for case in CASES:
            completed = WORK_DIR / case / f'seed_{seed}' / 'test_metrics.json'
            if completed.exists() and not FORCE_RERUN:
                print('SKIP completed:', case, 'seed', seed)
                continue
            command = [TRAIN_PYTHON, str(CODE_DIR / 'run_experiment.py'),
                       '--case', case, '--data-yaml', str(DATA_YAML),
                       '--work-root', str(WORK_DIR), '--pretrained', PRETRAINED,
                       '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
                       '--workers', str(WORKERS), '--image-size', str(IMAGE_SIZE),
                       '--seed', str(seed), '--patience', str(PATIENCE)]
            print('RUN:', case, 'seed', seed)
            subprocess.run(command, check=True)


## 6. Gom kết quả
`collect_results.py` gom 15 run (5 case × 3 seed) thành CSV thô (`RESULT_CSV`) và CSV
tổng hợp mean±std theo seed (`*_aggregate.csv`), có `delta_bbox_mAP_mean` của từng biến
thể ASRM so với `baseline_p2`.

In [ ]:
subprocess.run([TRAIN_PYTHON, str(CODE_DIR / 'collect_results.py'),
                '--work-root', str(WORK_DIR), '--seeds', *map(str, SEEDS),
                '--cases', *CASES, '--output', str(RESULT_CSV)],
               check=True)
import pandas as pd
from IPython.display import display
results = pd.read_csv(RESULT_CSV)
display(results)
aggregate_csv = RESULT_CSV.with_name(RESULT_CSV.stem + '_aggregate' + RESULT_CSV.suffix)
aggregate = pd.read_csv(aggregate_csv)
display(aggregate)


In [ ]:
# Tùy chọn: chỉ upload CSV khi bạn chủ động bật cờ và điền write token.
if UPLOAD_RESULTS:
    assert HF_WRITE_TOKEN and not HF_WRITE_TOKEN.startswith('PASTE_')
    from huggingface_hub import HfApi
    api = HfApi(token=HF_WRITE_TOKEN)
    api.upload_file(path_or_fileobj=str(RESULT_CSV), repo_id=HF_REPO, repo_type='dataset',
                     path_in_repo='results/comparison_yolo.csv')
    api.upload_file(path_or_fileobj=str(aggregate_csv), repo_id=HF_REPO, repo_type='dataset',
                     path_in_repo='results/comparison_yolo_aggregate.csv')
    print('Uploaded', RESULT_CSV, 'and', aggregate_csv)
